# Metrics from Joined Datasets

TL;DR: In a SLayer query, you can include not only dimensions, but also metrics from linked datasets, using the usual dot syntax. 

When you query a metric from a joined dataset, SLayer computes it in a separate sub-query to avoid row multiplication from the JOIN. This guarantees correctness — the metric value is the same whether you query it from the joined dataset or from the population directly.

This notebook demonstrates cross-dataset metrics with working code.

See also: [Cross-Dataset Metrics](../../concepts/queries.md#cross-dataset-metrics) | [Relationships](../../concepts/datasets.md#relationships) | [Expressions](../../concepts/expressions.md)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()

## The Problem: JOINs Tie Metrics to the Wrong Grain

Suppose you want monthly order revenue alongside a **total customer base count** — a single global figure shown on every row, not "customers who ordered that month." Orders → customers is many-to-one (each order has exactly one customer), so a naive `JOIN orders → customers` does *not* inflate `SUM(order_total)`. But aggregating customer count after that join scopes the metric to the order/month grain — you'd end up counting the customers who placed an order that month, not the global customer base.

(In other join shapes — say a one-to-many like orders → items — the naive join *does* multiply rows and inflate sums. The same isolation strategy below handles both cases.)

SLayer avoids this by computing the cross-dataset metric in a separate sub-query, grouped only by dimensions shared between the two datasets, then joining the result back. Orders and customers don't share a time or grouping dimension here, so the customers sub-query produces a single row — the global customer count — broadcast across every month via `CROSS JOIN`. The order metrics stay correct, and the customer metric stays at its intended global grain.

In [2]:
orders_dataset = next(d for d in datasets if d.name == "orders")

print("orders dataset relationships:")
for r in orders_dataset.relationships:
    print(f"  -> {r.to}  ON {r.column_pairs}")

customers_dataset = next(d for d in datasets if d.name == "customers")
print(f"\ncustomers metrics: {[m.name for m in customers_dataset.metrics]}")

stores_dataset = next(d for d in datasets if d.name == "stores")
print(f"stores metrics: {[m.name for m in stores_dataset.metrics]}")

orders dataset relationships:
  -> customers  ON [('customer_id', 'id')]
  -> stores  ON [('store_id', 'id')]

customers metrics: ['customer_count']
stores metrics: ['store_count']


## Cross-Dataset Metric Syntax

To reference a metric from a joined dataset, use dotted syntax in the metric expression: `"joined_dataset.metric_name"`. For example, `"count(customers.*)"` computes `COUNT(*)` on the customers dataset, scoped to whatever dimensions the source and target share.

The population must have an explicit [relationship](../../concepts/datasets.md#relationships) to the target dataset.

In [3]:
# Monthly orders alongside the global customer base count.
# `count(customers.*)` is a cross-dataset metric: SLayer computes it in a
# separate sub-query so the LEFT JOIN never multiplies the order rows.
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "orders.ordered_at", "granularity": "month"}],
        "metrics": [
            "count(orders.*)",
            "sum(orders.order_total)",
            "count(customers.*)",
        ],
        "order": [{"field": "orders.ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

print(f"{'Month':<12} {'Orders':>8} {'Revenue':>14} {'Customers':>10}")
print("-" * 48)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    count = row["orders._count"]
    rev = row["orders.order_total_sum"]
    custs = row["customers._count"]
    print(f"{month:<12} {count:>8,} ${rev:>13,.2f} {custs:>10,}")

Month          Orders        Revenue  Customers
------------------------------------------------
2023-11           166 $     2,309.68        104
2023-12         1,823 $    20,466.58        163
2024-01         2,121 $    25,481.00        187
2024-02         2,731 $    31,736.94        238
2024-03         3,733 $    40,719.67        283
2024-04         3,903 $    44,489.03        314


In [4]:
print("Generated SQL:")
print(result.sql)

Generated SQL:
WITH _base AS (
  SELECT
    CAST(DATE_TRUNC('MONTH', CAST(orders.ordered_at AS TIMESTAMP)) AS DATE) AS "orders.ordered_at",
    COUNT(*) AS "orders._count",
    SUM(orders.order_total) AS "orders.order_total_sum"
  FROM orders AS orders
  GROUP BY
    CAST(DATE_TRUNC('MONTH', CAST(orders.ordered_at AS TIMESTAMP)) AS DATE)
), _cm_orders__customers___count AS (
  SELECT
    _base."orders__ordered_at_month" AS "orders__ordered_at_month",
    COUNT(*) AS "_count_host"
  FROM (
    SELECT
      CAST(DATE_TRUNC('MONTH', CAST(orders.ordered_at AS TIMESTAMP)) AS DATE) AS "orders__ordered_at_month",
      customers.id AS _ek0
    FROM customers AS customers
    LEFT JOIN orders AS orders
      ON customers.id = orders.customer_id
    WHERE
      NOT customers.id IS NULL
    GROUP BY
      CAST(DATE_TRUNC('MONTH', CAST(orders.ordered_at AS TIMESTAMP)) AS DATE),
      customers.id
  ) AS _base
  GROUP BY
    _base."orders__ordered_at_month"
)
SELECT
  _base."orders.ordered_at",
  

Notice the SQL: `count(customers.*)` was computed in a **separate CTE** (`_cm_orders__customers___count`) that selects from `customers` independently of orders. Because orders and customers share no time or grouping dimension, the CTE produces a single row — the global customer count — which is `CROSS JOIN`ed back to the per-month order aggregation.

The point: the order metrics are computed without joining customers in, so the join can never multiply the order rows. If both datasets did share a grouping dimension (e.g., a `signed_up_at` on customers truncated to month), the CTE would group by that dimension and the join would be a regular LEFT JOIN on it — same correctness guarantee, varying values per row.

## Transforms on Cross-Dataset Metrics

Window functions like `cumsum()`, `change()`, and `rank()` work on cross-dataset metrics. The transform is applied **after** the cross-dataset sub-query is joined back:

In [5]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "orders.ordered_at", "granularity": "month"}],
        "metrics": [
            "count(customers.*)",
            {"metric": "cumsum(count(customers.*))", "alias": "cumulative_customers"},
        ],
        "order": [{"field": "orders.ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

print(f"{'Month':<12} {'Customers':>10} {'Cumulative':>12}")
print("-" * 36)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    custs = row["customers._count"]
    cum = row["cumulative_customers"]
    print(f"{month:<12} {custs:>10,} {cum:>12,}")

Month         Customers   Cumulative
------------------------------------
2023-11             104          104
2023-12             163          267
2024-01             187          454
2024-02             238          692
2024-03             283          975
2024-04             314        1,289


## Summary

| Feature | Description |
|---------|-------------|
| **Dotted syntax** | `"customers.count"` references a metric on a joined dataset |
| **Sub-query isolation** | Cross-dataset metrics are computed in a separate CTE, avoiding row multiplication |
| **Shared dimensions** | The sub-query groups by dimensions shared between the population and target datasets |
| **LEFT JOIN** | The CTE result is joined back to the main query, preserving all main-query rows |
| **Transforms** | `cumsum()`, `change()`, `rank()` etc. work on cross-dataset metrics |

See the [Cross-Dataset Metrics docs](../../concepts/queries.md#cross-dataset-metrics) and [Expressions docs](../../concepts/expressions.md) for the full reference.